# 2. XGBoost baseline

I use five jet summaries to see how far I get without individual cells.
I compare unpaired jets, which are enriched in beam-induced background (BIB),
with simulated jets matched to long-lived particles (LLPs). Choose **PPP
first look** and run from the top.

## How one tree split works

Each tree adds its leaf value to the logit: $z_i\leftarrow z_i+\nu f_m(x_i)$,
where $\nu$ is the learning rate. The BIB score is $s_i=1/(1+e^{-z_i})$.
For weighted binary log loss the slope and curvature are
$g_i=w_i(s_i-y_i)$ and $h_i=w_i s_i(1-s_i)$. I use labels 1 for unpaired and
0 for LLP. A missed unpaired jet has $g_i<0$, so raising its logit lowers the loss.

For a leaf $I$ let $G_I=\sum_{i\in I}g_i$ and $H_I=\sum_{i\in I}h_i$.
Minimizing $G_Ia+(H_I+\lambda)a^2/2$ gives the leaf value
$a=-G_I/(H_I+\lambda)$. Splitting parent $P$ into $L,R$ gives
$$\mathrm{gain}=\frac12\left(\frac{G_L^2}{H_L+\lambda}
+\frac{G_R^2}{H_R+\lambda}-\frac{G_P^2}{H_P+\lambda}\right)-\gamma,$$
where $\lambda$ shrinks leaf values and $\gamma$ charges for adding a leaf.

For example, take LLP jets at 0 and 1 ns and unpaired jets at 5 and 7 ns.
With unit weights and initial scores 0.5, a cut at 3 ns gives
$G_L=1$, $G_R=-1$ and $H_L=H_R=0.5$. For $\lambda=1$, $\gamma=0$,
the gain is $2/3$. A cut that puts one jet from each class on each side
has gain 0. With $\nu=0.3$, the selected tree moves the scores to about
0.45 and 0.55. The next tree recomputes the gradients from those scores.
[XGBoost model derivation](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)

## Read the jets

The main inputs are the ATLAS time, the time spread and the time uncertainty
(all in ns), the cluster count and the Tile energy fraction. I compare with
a model that sees only momentum across the beam ($p_T$) and direction ($|\eta|$).

I flatten each event into one row per jet and keep its event ID for splitting.
I keep matched LLP jets and require valid momentum, direction and time in
both samples, with $p_T\ge30$ GeV, $|\eta|<1.7$ and $0<f_{\rm Tile}<1$.
The Tile cut requires energy in both calorimeters.
[Cells and selections](01_cells_and_selections.ipynb)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir())
data_folder = project / "DataForEmre"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
run_dir = project / "analysis/outputs/notebook_03_xgboost" / stamp
run_dir.mkdir(parents=True, exist_ok=False)

In [ ]:
physics_features = ["jet_time_ATLAS", "jet_time_spread", "jet_time_sigma",
                    "jet_n_clusters", "jet_TileEnergyFrac"]
jet_fields = ["jet_pt", "jet_eta", *physics_features]
event_fields = ["runNumber", "eventNumber"]

sources = {
    "LLP": ("collision/LLP/N1N2_rpv_1400_10ns", 0),
    "unpaired": ("non_collision/unpaired/physics_Background_NTuple_unsplit_v2", 1),
}
source_files = {sample: sorted((data_folder / folder).glob("*.root"))
                for sample, (folder, _) in sources.items()}
assert all(source_files.values()), "Check the two data folders under DataForEmre."

In [ ]:
def jet_table(batch, keys, first_row):
    counts = ak.to_numpy(ak.num(batch["jet_pt"], axis=1))
    columns = {}
    for field in jet_fields:
        observed = ak.to_numpy(ak.num(batch[field], axis=1))
        np.testing.assert_array_equal(observed, counts)
        values = ak.to_numpy(ak.flatten(batch[field], axis=1))
        columns[field] = np.ma.asarray(values, dtype=float).filled(np.nan)
    table = pd.DataFrame(columns).replace([-999999.0, np.inf, -np.inf], np.nan)
    table["event_group"] = np.repeat(keys, counts)
    table["row_index"] = np.repeat(np.arange(first_row, first_row + len(batch)), counts)
    table["jet_index"] = ak.to_numpy(ak.flatten(ak.local_index(batch["jet_pt"], axis=1)))
    table["jet_pt_GeV"] = table["jet_pt"] / 1000
    table["jet_abs_eta"] = table["jet_eta"].abs()
    return table

In [ ]:
def mark_missing_inputs(table):
    spread = table["jet_time_spread"]
    uncertainty = table["jet_time_sigma"]
    clusters = table["jet_n_clusters"]
    table.loc[spread.lt(0), "jet_time_spread"] = np.nan
    table.loc[uncertainty.le(0), "jet_time_sigma"] = np.nan
    invalid_clusters = clusters.lt(1) | clusters.ne(np.floor(clusters))
    table.loc[invalid_clusters, "jet_n_clusters"] = np.nan

In [ ]:
def select_batch(table, batch, sample):
    truth = np.ones(len(table), dtype=bool)
    if sample == "LLP":
        truth = ak.to_numpy(ak.flatten(ak.fill_none(batch["jet_is_llp_matched"], 0))) == 1
    tile = table["jet_TileEnergyFrac"]
    cuts = {
        "LLP match": truth,
        "Valid pT, eta and time": table[["jet_pt_GeV", "jet_eta", "jet_time_ATLAS"]].notna().all(axis=1),
        "pT >= 30 GeV": table["jet_pt_GeV"].ge(30),
        "abs(eta) < 1.7": table["jet_abs_eta"].lt(1.7),
        "Energy in both calorimeters": tile.gt(0) & tile.lt(1),
    }
    keep = np.ones(len(table), dtype=bool)
    flow = {"All jets": len(table)}
    for name, condition in cuts.items():
        keep &= np.asarray(condition, dtype=bool)
        flow[name] = int(keep.sum())
    return table.loc[keep], flow

In [ ]:
def read_source(path, sample, label, seen_events):
    fields = [*event_fields, *jet_fields]
    if sample == "LLP":
        fields.append("jet_is_llp_matched")
    first_row = 0
    with uproot.open(path) as source:
        tree = source["JetContent"]
        for batch in tree.iterate(fields, step_size=10_000, library="ak"):
            keys = [f"{sample}:{run}:{event}" for run, event in
                    zip(batch["runNumber"], batch["eventNumber"])]
            assert len(set(keys)) == len(keys) and seen_events.isdisjoint(keys), path
            seen_events.update(keys)
            table = jet_table(batch, keys, first_row)
            mark_missing_inputs(table)
            selected, counts = select_batch(table, batch, sample)
            selected = selected.assign(sample=sample, label=label,
                                       source_file=str(path.relative_to(data_folder)))
            yield selected, counts
            first_row += len(batch)

In [ ]:
parts, cut_rows = [], []
for sample, files in source_files.items():
    seen_events = set()
    for path in files:
        for selected, counts in read_source(path, sample, sources[sample][1], seen_events):
            parts.append(selected)
            cut_rows.append({"sample": sample, **counts})
jets = pd.concat(parts, ignore_index=True)
assert not jets.duplicated(["event_group", "jet_index"]).any()
cutflow = pd.DataFrame(cut_rows).groupby("sample").sum().T
display(cutflow)
cutflow.to_csv(run_dir / "cutflow.csv")

## Train and set a cut

I split events 60/20/20 so jets from one event stay together. I give each
training jet weight $N/(2N_c)$, where $N_c$ is its class count. Each class
then has the same total weight. Validation picks the boosting round and the cut.

I tag a jet when its score is at least the cut. I sort the validation LLP scores
and place the cut just above the $\lceil 0.95N_{\rm LLP}\rceil$-th score from the
bottom. That keeps at least 95% of LLP jets, including ties at that score.

In [ ]:
seed = 42
events = jets[["event_group", "label"]].drop_duplicates().sort_values("event_group")
train_val, test_events = train_test_split(
    events, test_size=0.20, random_state=seed, stratify=events["label"],
)
train_events, val_events = train_test_split(
    train_val, test_size=0.25, random_state=seed, stratify=train_val["label"],
)
split_frames = {"train": train_events, "validation": val_events, "test": test_events}
assignment = {group: split for split, frame in split_frames.items()
              for group in frame["event_group"]}
jets["split"] = jets["event_group"].map(assignment)
assert jets["split"].notna().all()
assert jets.groupby("event_group")["split"].nunique().eq(1).all()
table = jets.groupby(["split", "sample"]).agg(jets=("label", "size"), events=("event_group", "nunique"))
display(table)

In [ ]:
model_features = {
    "XGBoost": physics_features,
    "Kinematics only": ["jet_pt_GeV", "jet_abs_eta"],
}
parameters = dict(n_estimators=600, learning_rate=0.05, max_depth=3,
                  min_child_weight=10, subsample=0.8, reg_lambda=5.0,
                  objective="binary:logistic", eval_metric="logloss",
                  tree_method="hist", n_jobs=4, random_state=seed, base_score=0.5)

def fit_trees(train, validation, features):
    weights = len(train) / (2 * train["label"].value_counts())
    stop = EarlyStopping(rounds=40, data_name="validation_1", metric_name="logloss", save_best=True)
    model = XGBClassifier(**parameters, callbacks=[stop])
    model.fit(train[features], train["label"], sample_weight=train["label"].map(weights),
              eval_set=[(train[features], train["label"]), (validation[features], validation["label"])],
              sample_weight_eval_set=[train["label"].map(weights), validation["label"].map(weights)],
              verbose=False)
    return model

In [ ]:
def validation_cut(model, validation, features):
    signal = validation.loc[validation["label"].eq(0), features]
    score = np.sort(model.predict_proba(signal)[:, 1].astype(np.float64))
    keep_count = int(np.ceil(0.95 * len(score)))
    return float(np.nextafter(score[keep_count - 1], np.inf))

train = jets.loc[jets["split"].eq("train")]
validation = jets.loc[jets["split"].eq("validation")]
test = jets.loc[jets["split"].eq("test")].copy()
models, cuts = {}, {}
for name, features in model_features.items():
    models[name] = fit_trees(train, validation, features)
    cuts[name] = validation_cut(models[name], validation, features)
    print(name, "kept rounds:", models[name].best_iteration + 1, "cut:", round(cuts[name], 5))

## Count test jets

I show the counts for all jet times and again for delayed jets (ATLAS time
above 3 ns). AUC measures how well the scores rank the two samples. The raw
counts show the performance at the fixed validation cut.

In [ ]:
def score_metrics(labels, scores, cut):
    labels = np.asarray(labels)
    unpaired = labels == 1
    tagged = scores >= cut
    rejected = int((tagged & unpaired).sum())
    retained = int((~tagged & ~unpaired).sum())
    return {
        "Unpaired tagged": rejected, "Unpaired total": int(unpaired.sum()),
        "LLP kept": retained, "LLP total": int((~unpaired).sum()),
        "ROC AUC": roc_auc_score(labels, scores) if np.unique(labels).size == 2 else np.nan,
    }

def test_rows(frame, scores, cut, name, selection):
    regions = {"All times": np.ones(len(frame), dtype=bool),
               "ATLAS time > 3 ns": frame["jet_time_ATLAS"].gt(3).to_numpy()}
    return [{"Model": name, "Sample": selection, "Region": region,
             **score_metrics(frame.loc[mask, "label"], scores[mask], cut)}
            for region, mask in regions.items()]

In [ ]:
results_rows = []
predictions = test[["source_file", "row_index", "jet_index", "event_group", "label"]].copy()
for name, features in model_features.items():
    scores = models[name].predict_proba(test[features])[:, 1].astype(np.float64)
    predictions[name] = scores
    results_rows.extend(test_rows(test, scores, cuts[name], name, "Broad"))
display(pd.DataFrame(results_rows).round(4))

## Repeat with similar momentum and direction

The samples have different momentum and direction. I bin $p_T$ and $|\eta|$,
then draw the same number from each class within each bin and split:
$m_b=\min(n_{\rm LLP,b},n_{\rm unpaired,b})$. I use bins that have at least three
jets from each class in training. I keep the original event assignments, then
refit the trees and cuts with these smaller samples.

In [ ]:
pt_edges = np.geomspace(30, 3000, 22)
eta_edges = np.linspace(0, 1.7, 10)
binned = jets.assign(pt_bin=pd.cut(jets["jet_pt_GeV"], pt_edges, labels=False, include_lowest=True),
                     eta_bin=pd.cut(jets["jet_abs_eta"], eta_edges, labels=False, include_lowest=True))
bin_columns = ["pt_bin", "eta_bin"]
train_counts = (binned.loc[binned["split"].eq("train")]
                .groupby([*bin_columns, "label"]).size().unstack(fill_value=0))
supported = train_counts.index[train_counts.min(axis=1).ge(3)]
in_support = pd.MultiIndex.from_frame(binned[bin_columns]).isin(supported)
matched_parts = []
for _, group in binned.loc[in_support].groupby(["split", *bin_columns]):
    count = group["label"].value_counts().reindex([0, 1], fill_value=0).min()
    if count:
        matched_parts.append(group.groupby("label").sample(n=count, random_state=seed))
matched = pd.concat(matched_parts).sort_index()
balance = matched.groupby(["split", *bin_columns, "label"]).size().unstack(fill_value=0)
assert balance[0].eq(balance[1]).all()
display(matched.groupby(["split", "sample"]).size().unstack())

In [ ]:
matched_train = matched.loc[matched["split"].eq("train")]
matched_validation = matched.loc[matched["split"].eq("validation")]
matched_test = matched.loc[matched["split"].eq("test")].copy()
matched_predictions = predictions.loc[matched_test.index, ["source_file", "row_index", "jet_index",
                                                           "event_group", "label"]].copy()
matched_models, matched_cuts = {}, {}
for name in ["XGBoost", "Kinematics only"]:
    features = model_features[name]
    model = fit_trees(matched_train, matched_validation, features)
    cut = validation_cut(model, matched_validation, features)
    scores = model.predict_proba(matched_test[features])[:, 1].astype(np.float64)
    matched_predictions[name] = scores
    matched_models[name], matched_cuts[name] = model, cut
    results_rows.extend(test_rows(matched_test, scores, cut, name, "Matched"))
summary = pd.DataFrame(results_rows)
display(summary.loc[summary["Sample"].eq("Matched")].round(4))

## Check and save the run

I keep the round with the lowest validation loss (the dashed line). The
scores, settings and saved trees go into the run directory printed below.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for axis, model, title in zip(axes, [models["XGBoost"], matched_models["XGBoost"]],
                              ["Broad selection", "Matched kinematics"]):
    curves = model.evals_result()
    for key, split in [("validation_0", "Train"), ("validation_1", "Validation")]:
        loss = curves[key]["logloss"]
        axis.plot(np.arange(1, len(loss) + 1), loss, label=split)
    axis.axvline(model.best_iteration + 1, color="grey", linestyle="--", label="Kept round")
    axis.set(title=title, xlabel="Boosting round", ylabel="Weighted log loss")
    axis.legend()
fig.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
for label, study_models in [("broad", models), ("matched", matched_models)]:
    for name, model in study_models.items():
        filename = label + "_" + name.lower().replace(" ", "_") + ".ubj"
        model.save_model(run_dir / filename)
settings = {"seed": seed, "features": model_features, "parameters": parameters,
            "broad_cuts": cuts, "matched_cuts": matched_cuts,
            "pt_edges_GeV": pt_edges.tolist(), "abs_eta_edges": eta_edges.tolist(),
            "supported_bins": supported.to_frame(index=False).to_dict(orient="records")}
(run_dir / "settings.json").write_text(json.dumps(settings, indent=2))
summary.to_csv(run_dir / "test_summary.csv", index=False)
jets[["event_group", "label", "split"]].drop_duplicates().to_csv(run_dir / "event_split.csv.gz", index=False)
predictions.to_csv(run_dir / "broad_scores.csv.gz", index=False)
matched_predictions.to_csv(run_dir / "matched_scores.csv.gz", index=False)
print("New results:", run_dir)

In the saved run, the kinematics-only AUC drops from about 0.99 to 0.59 after
matching. The five-input model gets about 0.90 in that matched sample. At the
validation cut, it tags **0/18** delayed unpaired jets and keeps **104/104**
delayed LLP jets. I check those counts separately because the full AUC
includes early and in-time jets. [Train and compare](03_train_and_compare.ipynb)